# Train YOLO for COCO-Task Images

Run this notebook from a fresh Colab runtime. It trains one YOLO model to detect normal COCO objects in the COCO-task images. MiniLM handles task preference separately.

## 1. Runtime

Select `Runtime > Change runtime type > T4 GPU`, then run this cell.

In [ ]:
!nvidia-smi

## 2. Mount Drive and Set Paths

This cell keeps the current folder as `/content`. Do not change into a Drive folder manually.

In [ ]:
%cd /content

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/minilm-new-pipeline')
IMAGE_ROOT = Path('/content/drive/MyDrive/datasets/mscoco')
YOLO_DATASET_DIR = Path('/content/yolo_coco_tasks')
YOLO_RUNS_DIR = PROJECT_DIR / 'runs' / 'detect'

TRAIN_JSONL = PROJECT_DIR / 'derived' / 'coco_tasks_train.jsonl'
VAL_JSONL = PROJECT_DIR / 'derived' / 'coco_tasks_test.jsonl'
CONVERTER = PROJECT_DIR / 'prepare_yolo_coco_tasks.py'

print('Project:', PROJECT_DIR)
print('Images:', IMAGE_ROOT)
print('YOLO dataset output:', YOLO_DATASET_DIR)

## 3. Check Required Files

If this cell fails, fix the folder/file it names before continuing.

In [ ]:
required_paths = [
    PROJECT_DIR,
    TRAIN_JSONL,
    VAL_JSONL,
    CONVERTER,
    IMAGE_ROOT / 'train2014',
    IMAGE_ROOT / 'val2014',
]

for path in required_paths:
    print(path, 'OK' if path.exists() else 'MISSING')
    assert path.exists(), f'Missing: {path}'

converter_text = CONVERTER.read_text(encoding='utf-8')
assert 'PureWindowsPath' in converter_text, 'prepare_yolo_coco_tasks.py is old. Upload the fixed version first.'
print('Converter version: OK')

## 4. Check Drive Can Read Images

If this cell shows an input/output or transport endpoint error, use `Runtime > Restart runtime`, then start again from cell 1.

In [ ]:
from itertools import islice

def show_first_images(folder, count=5):
    try:
        files = [p.name for p in islice(folder.glob('*.jpg'), count)]
    except OSError as exc:
        raise RuntimeError(
            f'Google Drive cannot read {folder}. Restart runtime and remount Drive.'
        ) from exc
    print(folder)
    print(files)
    assert files, f'No .jpg images found in {folder}'

show_first_images(IMAGE_ROOT / 'train2014')
show_first_images(IMAGE_ROOT / 'val2014')

## 5. Install Dependencies

In [ ]:
!pip install -q ultralytics tqdm pillow

## 6. Convert JSONL to YOLO Dataset

Expected output: nonzero images and boxes for train and val.

In [ ]:
!python "$CONVERTER" \
  --train-jsonl "$TRAIN_JSONL" \
  --val-jsonl "$VAL_JSONL" \
  --image-root "$IMAGE_ROOT" \
  --out-dir "$YOLO_DATASET_DIR" \
  --image-mode symlink

## 7. Verify Converted Dataset

In [ ]:
data_yaml = YOLO_DATASET_DIR / 'data.yaml'
assert data_yaml.exists(), data_yaml

train_labels = list((YOLO_DATASET_DIR / 'labels' / 'train').glob('*.txt'))
val_labels = list((YOLO_DATASET_DIR / 'labels' / 'val').glob('*.txt'))
print('train label files:', len(train_labels))
print('val label files:', len(val_labels))
assert train_labels, 'No train labels generated. Do not train yet.'
assert val_labels, 'No val labels generated. Do not train yet.'

print(data_yaml.read_text(encoding='utf-8')[:800])

## 8. Train YOLO

Start with 20 epochs. Increase later only after this first run completes correctly.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')
model.train(
    data=str(YOLO_DATASET_DIR / 'data.yaml'),
    epochs=20,
    imgsz=640,
    batch=16,
    device=0,
    project=str(YOLO_RUNS_DIR),
    name='yolo_coco_tasks',
    exist_ok=True,
)

## 9. Confirm Trained Weights

In [ ]:
best_weights = YOLO_RUNS_DIR / 'yolo_coco_tasks' / 'weights' / 'best.pt'
last_weights = YOLO_RUNS_DIR / 'yolo_coco_tasks' / 'weights' / 'last.pt'

print('best:', best_weights)
print('last:', last_weights)
assert best_weights.exists(), 'best.pt was not found.'